# Your First AutoGen Agent — A Beginner's Guide

This notebook walks you through the **simplest possible AutoGen program**: send one message to a language model and print the reply.

By the end you will understand:
- What Microsoft AutoGen is and why developers use it
- How to talk to an AI model from Python code
- Why the code uses `async` and `await`
- How Jupyter notebooks handle async differently from regular Python scripts

---

### What is Microsoft AutoGen?

**AutoGen** is an open-source framework from Microsoft for building applications that use AI agents. An *agent* is a piece of software that:
1. Receives a message (a task or question)
2. Thinks using a language model (like GPT)
3. Produces a response or takes an action

AutoGen handles the plumbing — API calls, message history, agent coordination — so you can focus on what the agents should *do*.

In this first notebook we only use **one agent and one message**. Later notebooks build up to multi-agent systems where several agents collaborate.

### The pipeline

```
Your question
    │
    ▼
┌─────────────┐
│ UserMessage │
└─────────────┘
    │
    ▼
┌────────────────────────────┐   request   ┌────────────┐
│ OpenAIChatCompletionClient │ ──────────▶ │ OpenAI API │
└────────────────────────────┘ ◀────────── └────────────┘
    │                           response
    ▼
┌──────────┐
│ Response │ ──▶ printed to you
└──────────┘
```

Just one agent, one message, one reply — no hand-offs yet. This is the baseline every later notebook builds on.

## Step 1 — Install the Required Packages

AutoGen v0.4+ is split into focused packages:

| Package | What it provides |
|---|---|
| `autogen-core` | Core types: messages, models, agents |
| `autogen-ext[openai]` | The OpenAI chat client (wraps the OpenAI API) |
| `python-dotenv` | Loads secrets from a `.env` file so they're never hardcoded |

Run the cell below once. The `#` keeps it from running accidentally on subsequent runs.

In [ ]:
# Uncomment and run this once to install the packages
# !pip install autogen-core autogen-ext[openai] python-dotenv

## Step 2 — Import Libraries

Here is what each import does:

| Import | Purpose |
|---|---|
| `load_dotenv` | Reads a `.env` file and puts its values into environment variables |
| `os` | Lets us read those environment variables with `os.getenv()` |
| `UserMessage` | The AutoGen type that represents a message from a human |
| `OpenAIChatCompletionClient` | An AutoGen wrapper around the OpenAI chat API |

In [1]:
from dotenv import load_dotenv
import os
from autogen_core.models import UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("Imports successful!")

Imports successful!


## Step 3 — Load Your API Key

We never paste API keys directly into code — if you share the file or commit it to git, the key is exposed.

Instead, store secrets in a file called `.env` in the same folder as this notebook:

```
OPENAI_API_KEY=sk-...
OPENAI_MODEL_NAME=gpt-4o-mini
```

`load_dotenv()` reads that file and makes the values available via `os.getenv()`. The `.env` file should be listed in your `.gitignore` so it is never committed.

In [2]:
load_dotenv()  # reads .env from the current directory

api_key    = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")  # fallback if not set

# Print True/False — never print the key itself!
print("API key loaded:", bool(api_key))
print("Model:", model_name)

API key loaded: True
Model: gpt-4o-mini


> **Checkpoint:** If `API key loaded:` prints `False`, create a `.env` file in the same folder as this notebook and add your key, then re-run this cell.

## Step 4 — Understand `async` and `await`

Before we write the main code, let's understand why AutoGen uses `async`.

### The problem: waiting for the internet

When your code calls the OpenAI API, it sends a request over the internet and then **waits** for a reply. That wait might be 1–5 seconds. If your code just sits there frozen, nothing else can happen in the meantime.

### The solution: async I/O

Python's `async`/`await` lets a function *pause* while waiting for a slow operation (like a network request) and allow other work to continue. It is like putting your name down at a restaurant instead of standing at the counter — you can do other things while you wait.

```python
# Without async — blocks the whole program while waiting
response = model_client.create([...])   # frozen for 2 seconds

# With async — yields control while waiting
response = await model_client.create([...])  # other tasks can run here
```

### The `async def` rule
If a function uses `await` inside it, the function itself must be declared `async def`. It's infectious: any function that calls an `async` function must also be `async`.

### Jupyter is already async-friendly
In a regular Python script you'd need `asyncio.run(main())` to start the event loop. In Jupyter, the kernel already runs an async event loop, so you can write `await` **directly in a code cell** — no wrapper needed.

## Step 5 — Create the Model Client

`OpenAIChatCompletionClient` is AutoGen's wrapper around the OpenAI chat API. You pass it:
- **`model`** — the model ID to use (e.g. `gpt-5-mini`)
- **`api_key`** — your OpenAI API key

The client manages the HTTP connection, retries, and response parsing for you.

In [3]:
model_client = OpenAIChatCompletionClient(
    model=model_name,
    api_key=api_key,
)

print(f"Client created for model: {model_name}")

Client created for model: gpt-4o-mini


## Step 6 — Understand `UserMessage`

AutoGen represents messages with typed objects rather than plain strings. This makes it easier to build complex multi-agent conversations later.

`UserMessage` represents a message from a human. It has two fields:

| Field | Purpose |
|---|---|
| `content` | The text of the message |
| `source` | A label identifying who sent it (used for conversation history) |

You pass a **list** of messages to `model_client.create()`. This mirrors the OpenAI chat format, where a conversation is a sequence of messages — which is why it's a list even when you only have one.

In [4]:
message = UserMessage(
    content="What is Microsoft AutoGen?",
    source="user"
)

print("Message created:")
print(f"  content : {message.content}")
print(f"  source  : {message.source}")

Message created:
  content : What is Microsoft AutoGen?
  source  : user


## Step 7 — Send the Message and Get a Response

`model_client.create()` sends the message list to the model and returns a response object. We `await` it because it makes a network call.

The response object has a `.content` attribute that holds the model's reply text.

> **Note:** This cell makes a real API call. It will take a few seconds and uses a small amount of your OpenAI credits.

In [5]:
response = await model_client.create([message])

print("=== Model Response ===")
print(response.content)

=== Model Response ===
Microsoft AutoGen refers to a suite of tools and functionalities developed by Microsoft to facilitate the automation of software development tasks using artificial intelligence, particularly through the integration of generative AI capabilities. While the specific features and components may evolve, the general aim of AutoGen is to streamline processes such as code generation, documentation, testing, and even deployment.

Key aspects of Microsoft AutoGen typically include:

1. **Code Generation**: AutoGen can automatically produce code snippets or whole functions based on natural language prompts or other inputs, making it easier for developers to implement features quickly.

2. **Documentation Generation**: It can help create documentation for codebases, generating comments and explanations based on the context in the code.

3. **Testing Automation**: AutoGen can help in writing unit tests or other testing scripts by analyzing the existing code and generating ap

### What does the response object contain?

Let's inspect it so you know what else is available.

In [6]:
print("Type:", type(response))
print("Attributes:", [a for a in dir(response) if not a.startswith("_")])

Type: <class 'autogen_core.models._types.CreateResult'>
Attributes: ['cached', 'construct', 'content', 'copy', 'dict', 'finish_reason', 'from_orm', 'json', 'logprobs', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'parse_file', 'parse_obj', 'parse_raw', 'schema', 'schema_json', 'thought', 'update_forward_refs', 'usage', 'validate']


## Step 8 — Close the Client

Always close the client when you're done. This releases the underlying HTTP connection pool cleanly.

Think of it like closing a database connection — not strictly required in a short script, but good practice and important in long-running applications.

In [7]:
await model_client.close()
print("Client closed.")

Client closed.


## Step 9 — The Complete Program

Here is everything assembled into a single reusable async function — exactly as it would look in a `.py` script.

The only difference from a script: Jupyter lets us call `await run()` directly instead of needing `asyncio.run(run())`.

In [ ]:
async def run(question: str) -> str:
    """Send a single question to the model and return the answer."""
    client = OpenAIChatCompletionClient(
        model=model_name,
        api_key=api_key,
    )
    response = await client.create([
        UserMessage(content=question, source="user")
    ])
    await client.close()
    return response.content

# Try it:
answer = await run("What is Microsoft AutoGen?")
print(answer)

### Script vs Notebook — the key difference

```python
# In a .py script you must start the event loop manually:
import asyncio
asyncio.run(run("What is AutoGen?"))

# In a Jupyter notebook the event loop is already running,
# so you just await directly:
await run("What is AutoGen?")
```

Both approaches work; Jupyter just makes it a little more concise.

## Step 10 — Exercises

### Exercise 1 — Ask your own question
Change the question in the cell below and run it.

In [ ]:
# Change this question to anything you like
my_question = "Explain what an AI agent is in two sentences."

print(await run(my_question))

### Exercise 2 — Ask multiple questions in a loop

Run three different questions and print each answer with a label.

In [ ]:
questions = [
    "What is an AI agent?",
    "What is the difference between AutoGen and LangChain?",
    "Give me one use case for a multi-agent AI system.",
]

for i, q in enumerate(questions, 1):
    print(f"--- Q{i}: {q} ---")
    print(await run(q))
    print()

### Exercise 3 — Change the model

Try a different model by passing the `model` argument directly. Compare the responses.

> **Cost note:** Larger models (e.g. `gpt-4o`) are more capable but cost more per token.

In [ ]:
async def run_with_model(question: str, model: str) -> str:
    client = OpenAIChatCompletionClient(model=model, api_key=api_key)
    response = await client.create([UserMessage(content=question, source="user")])
    await client.close()
    return response.content

question = "What is Microsoft AutoGen? Answer in one sentence."

for model in ["gpt-5-mini", "gpt-4o-mini"]:
    print(f"=== {model} ===")
    print(await run_with_model(question, model))
    print()

## Key Takeaways

| Concept | What it means |
|---|---|
| **AutoGen** | Microsoft's framework for building AI agent applications |
| **`OpenAIChatCompletionClient`** | AutoGen's wrapper that talks to the OpenAI API for you |
| **`UserMessage`** | A typed object representing a message from a human |
| **`async def`** | Declares a function that can pause while waiting (e.g. for a network call) |
| **`await`** | Pauses the function until the async operation completes |
| **`asyncio.run()`** | Starts the event loop in a `.py` script (not needed in Jupyter) |
| **`.env` file** | Stores secrets like API keys outside of your code |
| **`load_dotenv()`** | Loads `.env` variables so `os.getenv()` can read them |
| **`model_client.close()`** | Releases the HTTP connection — always call this when done |

---

### What's next?

This notebook sent a single hard-coded message to one model. The next step is to:
- **Accept user input** at runtime (see `autogen-agent-demo1.2.py`)
- **Use multiple agents** that collaborate on a task (see `autogen-agent-demo2-notebook.ipynb`)
- **Build agent chains** where the output of one agent feeds the next (see `autogen-agent-chain.ipynb`)